# Running a Simple Tick Data Server

*Python for Algorithmic Trading, Chapter 7: Working with Real-Time Data and Sockets*

> **Notebook 约定**
> - **Markdown 单元格**：书中正文
> - **代码单元格**：书中代码；代码内的 `#` 注释为书中编号说明（callout）的对应解释
> - **单元格下方的输出**：书中展示的运行结果（原样保留，**未在本 notebook 中实际执行**）

This section shows how to run a simple tick data server based on simulated financial instrument prices. The model used for the data generation is the geometric Brownian motion (without dividends) for which an exact Euler discretization is available, as shown in Equation 7-1. Here, $S$ is the instrument price, $r$ is the constant short rate, $\sigma$ is the constant volatility factor, and $z$ is a standard normal random variable. $\Delta t$ is the interval between two discrete observations of the instrument price.

**Equation 7-1. Euler discretization of geometric Brownian motion**

$$S_t = S_{t-\Delta t} \cdot \exp\left(\left(r - \frac{\sigma^2}{2}\right)\Delta t + \sigma \sqrt{\Delta t}\, z\right)$$

Making use of this model, "Sample Tick Data Server" (at the end of this notebook) presents a Python script that implements a tick data server using ZeroMQ and a class called `InstrumentPrice` to publish new, simulated tick data in a randomized fashion. The publishing is randomized in two ways. First, the stock price value is based on a Monte Carlo simulation. Second, the length of the time interval between two publishing events is randomized. The remainder of this section explains the major parts of the script in detail.

## 1. Imports and the PUB socket

The first part of the script does some imports, among other things, for the Python wrapper of ZeroMQ. It also instantiates the major objects needed to open a socket of `PUB` type:

In [ ]:
import zmq                                    # (1) imports the Python wrapper for the ZeroMQ library
import math
import time
import random

context = zmq.Context()                        # (2) a Context object is instantiated; it is the central object for socket communication
socket = context.socket(zmq.PUB)               # (3) the socket is defined based on the PUB socket type ("communication pattern")
socket.bind('tcp://0.0.0.0:5555')              # (4) bind to the local IP address (0.0.0.0 on Linux and Mac OS,
                                               #     127.0.0.1 on Windows) and port number 5555

## 2. The `InstrumentPrice` class

The class `InstrumentPrice` is for the simulation of instrument price values over time. As attributes, there are the major parameters for the geometric Brownian motion in addition to the instrument symbol and the time at which an instance is created. The only method `.simulate_value()` generates new values for the stock price given the time passed since it has been called the last time and a random factor:

In [ ]:
class InstrumentPrice(object):
    def __init__(self):
        self.symbol = 'SYMBOL'
        self.t = time.time()                   # (1) stores the time of the initialization
        self.value = 100.
        self.sigma = 0.4
        self.r = 0.01

    def simulate_value(self):
        ''' Generates a new, random stock price.
        '''
        t = time.time()                        # (2) current time is recorded when the method is called
        dt = (t - self.t) / (252 * 8 * 60 * 60)  # (3) dt: time interval between now and self.t, in (trading) year fractions
        dt *= 500                              # (4) scale dt (by an arbitrary factor) to get larger price movements
        self.t = t                             # (5) update self.t: reference point for the next call
        self.value *= math.exp((self.r - 0.5 * self.sigma ** 2) * dt +
                               self.sigma * math.sqrt(dt) * random.gauss(0, 1))  # (6) new price via Euler scheme for GBM
        return self.value

## 3. Main loop

The main part of the script consists of the instantiation of an object of type `InstrumentPrice` and an infinite `while` loop. During the `while` loop, a new instrument price gets simulated, and a message is created, printed, and sent via the socket. Finally, the execution pauses for a random amount of time:

> ⚠️ 这是一个**无限循环**，在 notebook 中直接运行会一直阻塞。请将完整脚本保存为 `TickServer.py`（见文末），在终端中运行 `python TickServer.py`。

In [ ]:
ip = InstrumentPrice()                         # (1) instantiate an InstrumentPrice object

while True:                                    # (2) start an infinite while loop
    msg = '{} {:.2f}'.format(ip.symbol, ip.simulate_value())  # (3) message text from symbol attribute + newly simulated price
    print(msg)                                 # (4) print the message str object to standard out
    socket.send_string(msg)                    # (5) also send it to subscribed sockets
    time.sleep(random.random() * 2)            # (6) pause for a random time (0 to 2 seconds),
                                               #     simulating random arrival of new tick data

SYMBOL 100.00
SYMBOL 99.65
SYMBOL 99.28
SYMBOL 99.09
SYMBOL 98.76
SYMBOL 98.83
SYMBOL 98.82
SYMBOL 98.92
SYMBOL 98.57
SYMBOL 98.81
SYMBOL 98.79
SYMBOL 98.80


Executing the script (in a terminal: `python TickServer.py`) prints out messages as shown in the output above.

At this point, it cannot yet be verified whether the script is also sending the same message via the socket bound to `tcp://0.0.0.0:5555` (`tcp://127.0.0.1:5555` on Windows). To this end, another socket subscribing to the publishing socket is needed to complete the socket pair.

> **Note**
> Often, the Monte Carlo simulation of prices for financial instruments relies on homogeneous time intervals (like "one trading day"). In many cases, this is a "good enough" approximation when working with, say, end-of-day closing prices over longer horizons. In the context of intraday tick data, the random arrival of the data is an important characteristic that needs to be taken into account. The Python script for the tick data server implements the random arrival times by randomized time intervals during which it pauses the execution.

## Sample Tick Data Server

The following is a script that runs a sample tick data server based on ZeroMQ. It makes use of Monte Carlo simulation for the geometric Brownian motion. 下面的单元格使用 `%%writefile` 将完整脚本写入 `TickServer.py`（只写文件，不会启动服务）：

In [ ]:
%%writefile TickServer.py
#
# Python Script to Simulate a
# Financial Tick Data Server
#
# Python for Algorithmic Trading
# (c) Dr. Yves J. Hilpisch
# The Python Quants GmbH
#
import zmq
import math
import time
import random

context = zmq.Context()
socket = context.socket(zmq.PUB)
socket.bind('tcp://0.0.0.0:5555')


class InstrumentPrice(object):
    def __init__(self):
        self.symbol = 'SYMBOL'
        self.t = time.time()
        self.value = 100.
        self.sigma = 0.4
        self.r = 0.01

    def simulate_value(self):
        ''' Generates a new, random stock price.
        '''
        t = time.time()
        dt = (t - self.t) / (252 * 8 * 60 * 60)
        dt *= 500
        self.t = t
        self.value *= math.exp((self.r - 0.5 * self.sigma ** 2) * dt +
                               self.sigma * math.sqrt(dt) * random.gauss(0, 1))
        return self.value


ip = InstrumentPrice()

while True:
    msg = '{} {:.2f}'.format(ip.symbol, ip.simulate_value())
    print(msg)
    socket.send_string(msg)
    time.sleep(random.random() * 2)

Writing TickServer.py
